In [3]:
!pip install elasticsearch

  Using cached elasticsearch-9.5.1-py3-none-any.whl.metadata (9.0 kB)
  Using cached elastic_transport-9.4.2-py3-none-any.whl.metadata (3.9 kB)
Using cached elasticsearch-9.5.1-py3-none-any.whl (1.0 MB)
Using cached elastic_transport-9.4.2-py3-none-any.whl (66 kB)

   ------------- -------------------------- 1/3 [elastic-transport]
   ------------- -------------------------- 1/3 [elastic-transport]
   ------------- -------------------------- 1/3 [elastic-transport]
   ------------- -------------------------- 1/3 [elastic-transport]
   -------------------------- ------------- 2/3 [elasticsearch]
   -------------------------- ------------- 2/3 [elasticsearch]
   -------------------------- ------------- 2/3 [elasticsearch]
   -------------------------- ------------- 2/3 [elasticsearch]
   -------------------------- ------------- 2/3 [elasticsearch]
   -------------------------- ------------- 2/3 [elasticsearch]
   -------------------------- ------------- 2/3 [elasticsearch]
   -----------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
#import necessary modules
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

In [13]:
# check if the elasticsearch container is running
!curl http://localhost:9200/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0
curl: (7) Failed to connect to localhost port 9200 after 2227 ms: Could not connect to server


In [15]:
# instantiate Elasticsearch client localhost
client = Elasticsearch("http://localhost:9200")

In [17]:
# data are in a jsonl file
# each line is a different json record
# read jsonl file line by line

all_json_data = [] # a list of json records
with open("IR2026/documents.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        json_data = json.loads(line)
        all_json_data.append(json_data)


In [19]:
# take a peek at the data
print(f" Datatype all_json_data: {type(all_json_data)}")
print(f" Datatype json_data: {type(json_data)}")
print(f" Length of the list: {len(all_json_data)}")
all_json_data[0]

#for jd in all_json_data:
#    print(f"Title: {jd['title']}, Runtime: {jd['runtime']}, Plot: {jd['plot']}, Keyscene: {jd['keyScene']} Genre: {jd['genre']}, Released: {jd['released']}")


 Datatype all_json_data: <class 'list'>
 Datatype json_data: <class 'dict'>
 Length of the list: 40221


{'_id': '11547',
 'title': 'Is there a word meaning "an unwanted eponym"?',
 'text': 'An eponym is one way to eternal (if posthumous) fame. But is there a word meaning an eponym someone would sooner not have? (One would presume that Captain Charles _Boycott_ , Mr Justice _Lynch_ , and Patrick _Hooligan_ would not appreciate their undying notoriety.)'}

In [ ]:
# delete the index anytime
client.indices.delete(index="my_movies")

In [21]:
# configure index properties
# create fields

mapping = {
    "mappings": {
        "properties": {
            "_id": {"type": "text"},
            "title": {"type": "integer"},
            "text": {"type": "text"},
            
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_documents', body=mapping)

ConnectionError: Connection error caused by: ConnectionError(Connection error caused by: NewConnectionError(<urllib3.connection.HTTPConnection object at 0x000002A556A442C0>: Failed to establish a new connection: [WinError 10061] Δεν ήταν δυνατή η δημιουργία σύνδεσης, επειδή ο υπολογιστής προορισμού την απέρριψε ενεργά))

In [ ]:
def insert_document(document):
    response = client.index(index="my_documents", body=document)
    return response

# populate index one document at a time
for document in all_json_data:
    response = insert_document(document)
    print(document)

print("Done indexing documents into `my_documents` index!")

In [ ]:
# count documents in a specific index..<
client.cat.count(index=['my_movies'], params={"format": "json"})

In [ ]:
# index using bulk helpers
from elasticsearch.helpers import bulk

documents = []
for doc in all_json_data:
    documents.append({
        "_index": "my_movies",
        "_source": doc  # αν το doc είναι dict
    })

# Indexing με helpers.bulk
bulk(client, documents, refresh=True)

print("Done indexing documents into `my_movies` index!")

In [ ]:
def pretty_search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            title = hit["_source"]["title"]
            runtime = hit["_source"]["runtime"]
            plot = hit["_source"]["plot"]
            keyScene = hit["_source"]["keyScene"]
            genre = hit["_source"]["genre"]
            released = hit["_source"]["released"]
            
            pretty_output = f"\nID: {id}\nScore: {score}\nTitle: {title}\nRuntime: {runtime}\nPlot: {plot}\nKeyScene: {keyScene}\nGenre: {genre}\nReleased: {released}"

            print(pretty_output)


In [ ]:
search_results = client.search(index="my_movies", query={"match_all": {}})

# Print search results
pretty_search_response(search_results)


In [ ]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"match": {"plot": "cannibal killer"}},
)

# Print search results
pretty_search_response(search_results)

#experiment with 'killer', 'serial killer', 'cannibal killer'

In [ ]:
print("English analyzer:")
response = client.indices.analyze(
    body={
        "analyzer": "english",
        "text": "The actors were running quickly across the stage!"
    }
)

for token in response["tokens"]:
    print(token["token"])

print("\nStandard analyzer:")
response = client.indices.analyze(
    body={
        "analyzer": "standard",
        "text": "The actors were running quickly across the stage!"
    }
)

for token in response["tokens"]:
    print(token["token"])    

In [ ]:
# delete the index anytime
client.indices.delete(index="my_movies")

In [ ]:
# set similarity function to vector space model (tfidf)
# set analyzer to english analyzer (stopword removal, stemming, puncuation removal, etc.) for indexing and querying

vsm_mapping = {
    "settings": {
        "number_of_shards": 1,
        "similarity": {
            "scripted_tfidf": {
                "type": "scripted",
                "script": {
                    "source": "double tf = Math.sqrt(doc.freq); double idf = Math.log((field.docCount+1.0)/(term.docFreq+1.0)) + 1.0; double norm = 1/Math.sqrt(doc.length); return query.boost * tf * idf * norm;"
                }
            }
        },        
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"
                },
                "default_search": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "plot": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "keyScene": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "runtime": {
                "type": "integer"
            },
            "genre": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "released": {
                "type": "date",
                "format": "yyyy"
            }
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_movies', body=vsm_mapping)

In [ ]:
# delete the index anytime
client.indices.delete(index="my_movies")

In [ ]:
# set similarity function to vector space model (tfidf)
# set analyzer to english analyzer (stopword removal, stemming, puncuation removal, etc.) for indexing and querying

vsm_optimized_mapping = {
    "settings": {
        "similarity": {
            "scripted_tfidf": {
                "type": "scripted",
                "script": {
                    "source": "double tf = Math.sqrt(doc.freq); double idf = Math.log((field.docCount+1.0)/(term.docFreq+1.0)) + 1.0; double norm = 1/Math.sqrt(doc.length); return query.boost * tf * idf * norm;"
                }
            }
        },        
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"
                },
                "default_search": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "copy_to": "allContent",
                "similarity": "scripted_tfidf"
            },
            "plot": {
                "type": "text",
                "copy_to": "allContent",
                "similarity": "scripted_tfidf"
            },
            "keyScene": {
                "type": "text",
                "copy_to": "allContent",                
                "similarity": "scripted_tfidf"
            },
            "runtime": {
                "type": "integer"
            },
            "genre": {
                "type": "text",
                "copy_to": "allContent",                
                "similarity": "scripted_tfidf"
            },
            "released": {
                "type": "date",
                "format": "yyyy"
            },
            "allContent": {
                "type": "text",
                "similarity": "scripted_tfidf"                
            }            
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_movies', body=vsm_optimized_mapping)


In [ ]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"match": {"allContent": "killer"}},
)

# Print search results
pretty_search_response(search_results)

In [ ]:
from pprint import pprint
# The search results are in the field 'hits':
print("Total articles found", search_results['hits']['total']) #total results count returned
# Another way to check how many results have been actually returned:
print("Results returned", len(search_results['hits']['hits'])) # the response was limited to top 10 results
# Let's access the first result:
pprint(search_results['hits']['hits'][0]) #use pprint for a more convenient display of hierarchical structure

In [ ]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"multi_match": {"query": "police", "fields": ["plot", "keyScene"]}},
)

# Print search results
pretty_search_response(search_results)

In [ ]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"multi_match": {"query": "police", "fields": ["plot", "keyScene^3"]}},
)

# Print search results
pretty_search_response(search_results)

In [ ]:
search_results = client.search(
    index="my_movies", query={"range": {"runtime": {"gte": 150}}}
)

# Print search results
pretty_search_response(search_results)



In [ ]:
search_results = client.search(
    index="my_movies", query={"prefix": {"title": {"value": "go"}}}
)

# Print search results
pretty_search_response(search_results)

In [ ]:
search_results = client.search(
    index="my_movies",
    query={
        "bool": {
            "must": [
                {"match": {"plot": "crime"}},
                {"match": {"keyScene": "character"}},
            ]
        }
    },
)

# Print search results
pretty_search_response(search_results)



In [ ]:
search_results = client.search(
    index="my_movies",
    query={
        "bool": {
            "should": [
                {"match": {"plot": "crime"}},
                {"match": {"keyScene": "character"}},
            ]
        }
    },
)

# Print search results
pretty_search_response(search_results)
